In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from lcdb.db import LCDB, ResultSet
from lcdb.db._learning_curves import LearningCurve
import scipy
import scipy.optimize
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from types import SimpleNamespace


# load sklearn regressors
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.model_selection import ShuffleSplit, train_test_split, cross_validate
from sklearn.metrics import get_scorer

# Get Data For Surrogate

In [ ]:
def load_workflow_results(wf: str, data_dir: Path) -> ResultSet:
    f = data_dir / f"{wf}.jsonl"
    if not f.exists():
        print(f"[WARN] Missing JSONL for workflow: {wf} (expected {f})")
        return None
    rs = ResultSet.read_jsonl(f)

    rs.datasets_sorted = sorted(rs.datasets)
    return rs


In [ ]:
rs_liblinear = load_workflow_results("lcdb.workflow.sklearn.LibLinearWorkflow", Path("../data"))
rs_knn = load_workflow_results("lcdb.workflow.sklearn.KNNWorkflow", Path("../data"))
print(rs_knn.datasets_sorted)

# Train Surrogate

In [ ]:
class SurrogateBuilder:

    def __init__(self, rs, ignore_nan_values=True, allow_preprocessed_pipelines=True):
        if len(rs.workflows) != 1:
            raise ValueError(f"ResultSet must contain exactly one workflow but has these: {rs.workflows}.")
        self.workflow = sorted(rs.workflows)[0]

        if len(rs.datasets) != 1:
            raise ValueError(f"ResultSet must contain exactly one dataset but has these: {rs.datasets}.")
        self.openmlid = sorted(rs.datasets)[0]
    
        # create rows based on mean curve on validation error rate
        rows = []
        self.allow_preprocessed_pipelines = allow_preprocessed_pipelines
        for result in rs:
            x = result["config"].copy()
            
            # cut out curves with non-standard preprocessing if not allowed
            if not allow_preprocessed_pipelines:
                if x["pp@cat_encoder"] != 'ordinal' or x["pp@decomposition"] != "none" or x["pp@featureselector"] != "none" or x["pp@scaler"] != "none" or x["pp@featuregen"] != "none":
                    continue
                for col in [c for c in x.keys() if c.startswith("pp@")]:
                    del x[col]

            lc = LearningCurve.from_json(result["learning_curve"])
            val_error_rate_curves = lc.values[0, 1]
            for anchor, mean_val in zip(lc.anchors_size, val_error_rate_curves.mean(axis=(0, 1, 2))):
                if np.isnan(mean_val) and ignore_nan_values:
                    continue
                x_at_anchor = x.copy()
                x_at_anchor["anchor"] = anchor
                x_at_anchor["score"] = mean_val
                rows.append(x_at_anchor)
        self.df = pd.DataFrame(rows)
    
    @property
    def anchors_in_original_data(self):
        return sorted(self.df["anchor"].unique())

    def get_standard_pipeline_for_model(self, model):
        """
        Create an sklearn Pipeline that applies:
        - OneHotEncoder to categorical columns
        - StandardScaler to numerical columns
        - model as the final estimator

        Parameters
        ----------
        model : sklearn estimator
            The model to be trained (e.g. LogisticRegression, RandomForestClassifier).
        dtypes : pandas.Series
            Column dtypes, typically X.dtypes.

        Returns
        -------
        sklearn.pipeline.Pipeline
        """

        if self.df is None:
            raise ValueError("Dataframe is not prepared. Call prepare_data() first.")
        dtypes = self.df.dtypes
        if not isinstance(dtypes, pd.Series):
            raise TypeError("dtypes must be a pandas Series (e.g., X.dtypes)")

        categorical_features = dtypes[dtypes == "object"].index.tolist()
        numeric_features = dtypes[dtypes != "object"].index.tolist()
        numeric_features.remove("score")

        preprocessor = ColumnTransformer(
            transformers=[
                #("num", StandardScaler(), numeric_features),
                ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
            ],
            remainder="passthrough"
        )

        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", model),
            ]
        )

        return pipeline

    def get_learning_surrogate_learning_curve(self, model_name, model, fixed_anchor=None, test_points=100, splits_per_anchor=5, schedule=None):

        # check whether we have the file
        filename = f"surrogate_lcs/{self.workflow}/{model_name}_{self.openmlid}_pp{self.allow_preprocessed_pipelines}_anchor{fixed_anchor}_{test_points}tp.json"

        # read from cache if possible
        path = Path(filename)

        # write results to cache
        if not path.exists():
        
            df = self.df.copy()
            if fixed_anchor is not None:
                df = df[df["anchor"] == fixed_anchor]

            # define standard schedule
            max_possible_num_training_points = len(df) - test_points
            if schedule is None:
                schedule = sorted(set([int(np.ceil(2**(i / 2))) for i in range(10, int(np.ceil(2 * np.log2(max_possible_num_training_points))))]))
                if max(schedule) < max_possible_num_training_points:
                    schedule.append(max_possible_num_training_points)
            
            if max(schedule) > max_possible_num_training_points:
                    raise ValueError(f"Not enough data points for the largest anchor in the schedule. Maximum number of  training points should be {max_possible_num_training_points}, but the schedule asks for (up to) {max(schedule)}.")
            
            # extract data
            X = df.drop(columns=["score"])
            y = df["score"].values

            # build learning curve
            train_lc_pairs = []
            val_lc_pairs = []
            pbar = tqdm(total = len(schedule) * splits_per_anchor)
            for anchor in schedule:
                train_lc_pairs_at_anchor = []
                val_lc_pairs_at_anchor = []

                for i_train, i_test in ShuffleSplit(n_splits=splits_per_anchor, train_size=anchor, test_size=test_points).split(X):
                    model.fit(X.iloc[i_train], y[i_train])
                    
                    # add train curve entry
                    y_true = y[i_train]
                    y_pred = model.predict(X.iloc[i_train])
                    train_lc_pairs_at_anchor.append([list(y_true), list(y_pred)])

                    # add validation curve entry
                    y_true = y[i_test]
                    y_pred = model.predict(X.iloc[i_test])
                    val_lc_pairs_at_anchor.append([list(y_true), list(y_pred)])

                    pbar.update(1)

                train_lc_pairs.append(train_lc_pairs_at_anchor)
                val_lc_pairs.append(val_lc_pairs_at_anchor)
            pbar.close()

            # write results to file
            path.parent.mkdir(parents=True, exist_ok=True)
            with open(filename, "w") as f:
                json.dump((schedule, train_lc_pairs, val_lc_pairs), f)


        with open(filename, "r") as f:
            return json.load(f)

        

In [ ]:
all_results = {}
failed = {}

for ds in rs_knn.datasets_sorted:
    print("dataset =", ds)
    try:
        sb_knn = SurrogateBuilder(
            rs=rs_knn.filter_datasets(ds),
            allow_preprocessed_pipelines=False
        )
        models_to_compare = {
            # "KNN-1": sb_knn.get_standard_pipeline_for_model(KNeighborsRegressor(n_neighbors=1)),
            # "KNN-5": sb_knn.get_standard_pipeline_for_model(KNeighborsRegressor(n_neighbors=5)),
            # "Linear Regression": sb_knn.get_standard_pipeline_for_model(LinearRegression()),
            # "Random Forest 1000": sb_knn.get_standard_pipeline_for_model(RandomForestRegressor(n_estimators=1000)),
            # "Extra Trees 1000": sb_knn.get_standard_pipeline_for_model(ExtraTreesRegressor(n_estimators=1000)),
            # "Gradient Boosting 100": sb_knn.get_standard_pipeline_for_model(GradientBoostingRegressor(n_estimators=100)),
            "Gradient Boosting 1000": sb_knn.get_standard_pipeline_for_model(GradientBoostingRegressor(n_estimators=1000))
        }
        
        anchors_in_original_data = sb_knn.anchors_in_original_data
        print(f"Will create surrogates for those different anchors in the original data: {anchors_in_original_data}")

        ground_truth_pairs_per_anchor = [
            {
                model_name: sb_knn.get_learning_surrogate_learning_curve(
                    model_name=model_name,
                    model=pl,
                    fixed_anchor=anchor_for_model,
                    test_points=10,
                    splits_per_anchor=20,
                    schedule=list(range(10, 101, 10))
                )
                for model_name, pl in models_to_compare.items()
            }
            for anchor_for_model in anchors_in_original_data
        ]

        ground_truth_pairs_across_anchors = {
            model_name: sb_knn.get_learning_surrogate_learning_curve(
                model_name=model_name,
                model=pl,
                test_points=100,
                splits_per_anchor=20,
                schedule=[10, 100, 500, 1000, 1400]
            )
            for model_name, pl in models_to_compare.items()
        }

        all_results[ds] = {
            "per_anchor": ground_truth_pairs_per_anchor,
            # "across_anchors": ground_truth_pairs_across_anchors
        }
        print(f"[OK] {ds}")

    except Exception as e:
        failed[ds] = repr(e)
        print(f"[FAIL] {ds} -> {e}")

print(f"done. ok={len(all_results)} failed={len(failed)}")

# for ds in rs_knn.datasets:
#     print("dataset =", ds)

#     sb_knn = SurrogateBuilder(rs=rs_knn.filter_datasets(ds), allow_preprocessed_pipelines=False)

#     ground_truth_pairs_per_anchor = [
#         {
#             model_name: 
#             sb_knn.get_learning_surrogate_learning_curve(
#                 model_name=model_name,
#                 model=pl,
#                 fixed_anchor=anchor_for_model,
#                 test_points=10,
#                 splits_per_anchor=20,
#                 schedule=list(range(10, 101, 10))
#             )
#             for model_name, pl in models_to_compare.items()
#         }
#         for anchor_for_model in anchors_in_original_data
#     ]

#     ground_truth_pairs_across_anchors = {
#         model_name: sb_knn.get_learning_surrogate_learning_curve(
#             model_name=model_name,
#             model=pl,
#             test_points=100,
#             splits_per_anchor=20,
#             schedule=[10, 100, 500, 1000, 1400]
#         )
#         for model_name, pl in models_to_compare.items()
#     }

In [ ]:
# # Logic of ground_truth_pairs_per_anchor
# model_name = "Linear Regression"
# idx_of_workflow_anchor = anchors_in_original_data.index(1024)
# idx_of_information = 1 # 0 for surrogate schedule, 1 for training data performances, 2 for validation data performances
# idx_of_surrogate_anchor = 4
# idx_of_repetition_seed = 0
# ground_truth_vs_predicted = 0 # 0 for ground truth, 1 for prediction
# idx_test_instance = 4

# print(ground_truth_pairs_per_anchor[idx_of_workflow_anchor][model_name][idx_of_information][idx_of_surrogate_anchor][idx_of_repetition_seed][ground_truth_vs_predicted][idx_test_instance])

# for data_for_seed in ground_truth_pairs_per_anchor[idx_of_workflow_anchor][model_name][2][0]: # validation data on first surrogate anchor
#     data_for_seed = np.array(data_for_seed)
#     if model_name == "Linear Regression":
#         assert np.all(data_for_seed[0] >= 0) # only check ground truth
#     else:
#         assert np.all(data_for_seed >= 0)

# Plot Surrogate LCs

In [ ]:
def plot_surrogate_learning_success(gt_vs_pred_data):
    fig, axs = plt.subplots(1, 2, figsize=(11, 3))

    for model_name, model_data in gt_vs_pred_data.items():
        surrogate_schedule = model_data[0]
        val_curve_gt_pred_pairs = np.array(model_data[2])
        
        # show learning curve of surrogate
        surrogate_lc = []
        for surrogate_anchor_index, pairs_at_trainig_anchor in enumerate(val_curve_gt_pred_pairs):
            rmses_at_anchor = []
            for fold in pairs_at_trainig_anchor:
                rmse_for_fold = np.sqrt(((fold[0] - fold[1])**2).mean())
                rmses_at_anchor.append(rmse_for_fold)
            surrogate_lc.append(rmses_at_anchor)
        surrogate_lc = np.array(surrogate_lc)
        mean = surrogate_lc.mean(axis=1)
        std = surrogate_lc.std(axis=1)
        axs[0].plot(surrogate_schedule, mean, marker="o", label=model_name)
        axs[0].fill_between(surrogate_schedule, mean - std, mean + std, alpha=0.2)


        # show comparison on last training anchor
        pairs_at_last_train_anchor = val_curve_gt_pred_pairs[-1].transpose(1, 2, 0).reshape(2, -1)

        axs[1].scatter(pairs_at_last_train_anchor[0], pairs_at_last_train_anchor[1], alpha=0.5, label=model_name)

    # plot settings
    ax = axs[0]
    ax.set_ylim([0, 0.05])
    ax.set_xlabel("Number of Training Points for Surrogate Model")
    ax.set_ylabel("Surrogate RMSE on Validation Set")

    ax = axs[1]
    ax.set_xlabel("True Validation Error Rate")
    ax.set_ylabel("Predicted Validation Error Rate")
    # ax.set_title(f"Surrogate Predictions vs Ground Truth (when trained with all data)", fontsize=10)
    ax.plot([0, 1], [0, 1], color='black', linestyle='--')

    for ax in axs:
        ax.legend()
        ax.grid()
    fig.suptitle(f"Surrogate Insights on Dataset {sb_knn.openmlid} for workflow {sb_knn.workflow} across anchors \n \n", fontsize=11)
    plt.show()

In [ ]:
def merge_per_anchor_for_plot(per_anchor_list):

    merged = {}
    model_names = sorted({mn for d in per_anchor_list for mn in d.keys()})

    for model_name in model_names:
        model_datas = [d[model_name] for d in per_anchor_list if model_name in d]
        if len(model_datas) == 0:
            continue

        schedule = model_datas[0][0]

        pairs_list = [np.array(md[2]) for md in model_datas]
        merged_pairs = np.concatenate(pairs_list, axis=1) 

        md0 = model_datas[0]
        middle = md0[1] if len(md0) > 1 else None  

        merged[model_name] = (schedule, middle, merged_pairs)

    return merged


# for ds in rs_knn.datasets_sorted:
#     if ds not in all_results:
#         continue

#     gt_vs_pred_data = merge_per_anchor_for_plot(all_results[ds]["per_anchor"])


#     plot_surrogate_learning_success(gt_vs_pred_data)


# plot_surrogate_learning_success(ground_truth_pairs_across_anchors)

## Extrapolation

In [ ]:
def get_num_par(model_id):
    if model_id == 'last1':
        return 1
    if model_id in ['pow2', 'log2', 'exp2', 'lin2', 'ilog2']:
        return 2
    if model_id in ['pow3', 'exp3', 'vap3', 'expp3', 'expd3', 'logpower3']:
        return 3
    if model_id in ['mmf4', 'wbl4', 'exp4', 'pow4', 'janoschek4']:
        return 4


def fit_model(sizes, scores, sizes_extrapolation, model_id, rep=5, verbose=True):
    sizes = np.array(sizes)
    scores = np.array(scores)

    bad_score = np.isnan(scores)

    sizes = sizes[bad_score == False]
    scores = scores[bad_score == False]

    # this defines the curve model
    def get_fun(beta):
        num_par = get_num_par(model_id)
        fun = None

        # unpack parameters
        if num_par == 1:
            a = beta[0]
        if num_par == 2:
            a, b = beta[0], beta[1]
        if num_par == 3:
            a, b, c = beta[0], beta[1], beta[2]
        if num_par == 4:
            a, b, c, d = beta[0], beta[1], beta[2], beta[3]

        # define curve models
        if model_id == 'pow2':
            fun = lambda x: -a * x ** (-b)
        if model_id == 'pow3':
            fun = lambda x: a - b * x ** (-c)
        if model_id == 'log2':
            fun = lambda x: -a * np.log(x) + b
        if model_id == 'exp3':
            fun = lambda x: a * np.exp(-b * x) + c
        if model_id == 'exp2':
            fun = lambda x: a * np.exp(-b * x)
        if model_id == 'lin2':
            fun = lambda x: a * x + b
        if model_id == 'vap3':
            fun = lambda x: np.exp(a + b / x + c * np.log(x))
        if model_id == 'mmf4':
            fun = lambda x: (a * b + c * x ** d) / (b + x ** d)
        if model_id == 'wbl4':
            fun = lambda x: (c - b * np.exp(-a * (x ** d)))
        if model_id == 'exp4':
            fun = lambda x: c - np.exp(-a * (x ** d) + b)
        if model_id == 'expp3':
            # fun = lambda x: a * np.exp(-b*x) + c
            fun = lambda x: c - np.exp((x - b) ** a)
        if model_id == 'pow4':
            fun = lambda x: a - b * (x + d) ** (-c)  # has to closely match pow3
        if model_id == 'ilog2':
            fun = lambda x: b - (a / np.log(x))
        if model_id == 'expd3':
            fun = lambda x: c - (c - a) * np.exp(-b * x)
        if model_id == 'logpower3':
            fun = lambda x: a / (1 + (x / np.exp(b)) ** c)
        if model_id == 'last1':
            fun = lambda x: (a + x) - x  # casts the prediction to have the correct size
        if model_id == 'janoschek4': 
            fun = lambda x: a - b * np.exp(-c * x ** d)
        return fun

    def objective(beta):  # this returns the residuals of the fit on the training points
        fun = get_fun(beta)
        return fun(sizes) - scores

    # we dp multiple repititions and collect best results in lists below
    beta_list = []
    trn_error = []

    # this model requires no optimization
    if model_id == 'last1':
        a = scores[-1]
        return np.array([a]), get_fun(np.array([a])), 0, 0

    # failure statistics
    #rep = 5
    fails_fit = 0
    fails_init = 0
    i = 0
    init = None

    while i <= rep:  # try repeatedly to fit a model
        num_par = get_num_par(model_id)

        beta = None
        error = True
        first = True
        # keep trying initial points until a suitable one is found
        while (error):

            if fails_init > 100 or fails_fit > 20:  # give up
                best_beta = np.zeros(num_par)
                # if verbose:
                #     print('giving up...')
                return best_beta, get_fun(best_beta), fails_init, fails_fit

            if not first:
                fails_init += 1
                # if verbose:
                    # print('initial value failed, retrying for ', model_id)
            init = np.random.rand(num_par)

            if model_id == 'pow4':  # this init works well for pow4
                best_beta, _, _, _ = fit_model(sizes, scores, sizes_extrapolation, 'pow3')
                init[0:3] = best_beta

            # check for errors in initial point
            trn_error_init = np.mean(objective(init) ** 2)
            fun_init = get_fun(init)
            sizes_all = np.hstack((sizes, sizes_extrapolation))
            hat_all = fun_init(sizes_all)
            nan_error1 = np.isnan(hat_all).any()
            inf_error1 = np.isinf(hat_all).any()
            nan_error2 = np.isnan(trn_error_init).any()
            inf_error2 = np.isinf(trn_error_init).any()
            error = nan_error1 or inf_error1 or nan_error2 or inf_error2

            first = False

        # start fitting
        beta = scipy.optimize.least_squares(objective, init, method="lm").x

        # check if fit extrapolates well to unseen sizes
        fun = get_fun(beta)
        extrapolations = fun(sizes_extrapolation)
        nan_error = np.isnan(extrapolations).any()
        inf_error = np.isinf(extrapolations).any()

        if nan_error or inf_error:
            pass  # redo's the optimization since extrapolations failed
            fails_fit += 1
            if verbose:
                print('fit failed, nan error?', nan_error, 'inf error?', inf_error, 'model?', model_id)
        else:
            i += 1
            pass  # save the parameter values and objective function
            beta_list.append(beta)
            trn_error.append(np.mean(objective(beta) ** 2))

    # select the best one
    trn_error = np.array(trn_error)
    best_i = np.argmin(trn_error)

    best_beta = beta_list[best_i]
    return best_beta, get_fun(best_beta), fails_init, fails_fit



def curves_models_fitting(lc_data, schedule, model_names, extrapolate,
                          mask_anchor_number=0, eval_anchor_number=0,
                          rep=10, verbose=False,
                          thr=0.05, max_x=1_000_000, n_extrap=300):
    fitting_results = []

    lc_data = np.asarray(lc_data, dtype=float)
    schedule = np.asarray(schedule, dtype=float)

    # remove nan in curve & align with schedule
    mask_indices = ~np.isnan(lc_data)
    scores = lc_data[mask_indices]
    schedule_obs = schedule[mask_indices]

    for model_name in model_names:
        try:
            # -------- training data (fix: mask_anchor_number=0 should keep all) --------
            if extrapolate and mask_anchor_number > 0:
                train_schedule = schedule_obs[:-mask_anchor_number]
                regress_target = scores[:-mask_anchor_number]
            else:
                train_schedule = schedule_obs
                regress_target = scores

            # -------- extrapolation schedule --------
            if extrapolate:
                x0 = float(schedule_obs.max())
                tail = np.geomspace(max(x0, 2.0), max_x, n_extrap)  
                schedule_extrap = np.unique(np.concatenate([schedule_obs, tail]))
                schedule_extrap.sort()
            else:
                schedule_extrap = schedule_obs

            # -------- fitting --------
            beta, model, fails_init, fails_fit = fit_model(
                train_schedule, regress_target,
                np.array(schedule_extrap),
                model_name, rep=rep, verbose=verbose
            )

            # predictions
            pred_obs = model(np.array(schedule_obs))
            pred_extrap = model(np.array(schedule_extrap))

            if extrapolate and eval_anchor_number > 0:
                mse = mean_squared_error(scores[-eval_anchor_number:], pred_obs[-eval_anchor_number:])
            else:
                mse = mean_squared_error(scores, pred_obs)

            # -------- first x where y<thr (only search after observed max) --------
            idx = np.where((pred_extrap < thr) & (schedule_extrap >= schedule_obs.max()))[0]
            x_at_thr = float(schedule_extrap[idx[0]]) if len(idx) > 0 else np.nan

            fitting_results.append({
                "schedule_obs": schedule_obs,
                "scores": scores,
                "pred_obs": pred_obs,
                "schedule_extrap": schedule_extrap,
                "pred_extrap": pred_extrap,
                "x_at_thr": x_at_thr,
                "thr": thr,
                "mse": mse,
                "curve_model": model_name,
                "beta": beta,
                "fails_init": fails_init,
                "fails_fit": fails_fit
            })

        except Exception as e:
            if verbose:
                print(f"Failed to fit model {model_name}. Error: {e}")

    return fitting_results


def extract_mean_rmse_and_schedule(model_data):
    schedule = np.array(model_data[0], dtype=float)
    val_curve_gt_pred_pairs = np.array(model_data[2])  

    rmses = []
    for pairs_at_training_anchor in val_curve_gt_pred_pairs:  #  schedule 
        fold_rmses = []
        for fold in pairs_at_training_anchor:                 #  fold
            fold_rmse = np.sqrt(((fold[0] - fold[1])**2).mean())
            fold_rmses.append(fold_rmse)
        rmses.append(fold_rmses)

    rmses = np.array(rmses)         
    mean_rmse = rmses.mean(axis=1)  
    return mean_rmse, schedule

In [ ]:
thr = 0.005
x_cap = 10000

for ds in rs_knn.datasets_sorted:
    if ds not in all_results:
        continue

    gt_vs_pred_data = merge_per_anchor_for_plot(all_results[ds]["per_anchor"])

    _real_show = plt.show
    plt.show = lambda *a, **k: None
    plot_surrogate_learning_success(gt_vs_pred_data)

    fig = plt.gcf()
    ax_left = fig.axes[0]

    for model_name, model_data in gt_vs_pred_data.items():
        curve_data, schedule = extract_mean_rmse_and_schedule(model_data)

        for curve_model in ["exp4", "pow4"]:
            fitting_results = curves_models_fitting(
                curve_data, schedule, [curve_model],
                extrapolate=True, mask_anchor_number=0, eval_anchor_number=0, rep=3,
                thr=thr, max_x=x_cap, n_extrap=400
            )
            result_df = pd.DataFrame(fitting_results)
            if result_df.empty:
                continue

            r = result_df.iloc[0]
            x = np.asarray(r["schedule_extrap"], dtype=float)
            y = np.asarray(r["pred_extrap"], dtype=float)
            x_hit = r["x_at_thr"]

            if np.isfinite(x_hit):
                m = x <= x_hit
                ax_left.plot(x[m], y[m], "--", label=f"{model_name} {curve_model}-fit")
                ax_left.axvline(x_hit, linestyle=":", alpha=0.8)
                ax_left.scatter([x_hit], [thr], s=40, zorder=5)
                ax_left.annotate(f"{curve_model} x@{thr:.3f}={x_hit:.0f}", (x_hit, thr),
                                 textcoords="offset points", xytext=(6, 6))
            else:
                m = x <= x_cap
                ax_left.plot(x[m], y[m], "--", label=f"{model_name} {curve_model}-fit (NO hit)")
                ax_left.axhline(thr, linestyle=":", alpha=0.8)

    ax_left.legend()

    plt.show = _real_show
    plt.show()
